In [8]:
import os
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

# --- 1. 초기 설정: 모델, 경로, 장치 설정 ---

print("="*50)
print("1. 초기 설정 시작...")

# Hugging Face 모델 및 라이브러리 로딩
from open_flamingo import create_model_and_transforms
from huggingface_hub import hf_hub_download

# 경로 설정
DEV_CSV_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/dev_test.csv"
SAMPLE_SUBMISSION_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/sample_submission.csv"
IMAGE_DIR = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images"
OUTPUT_PATH = "/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/submission/open_flamingo_2.csv"


1. 초기 설정 시작...


In [9]:

# 장치 설정 (GPU 우선 사용)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

# --- 2. 모델 로딩 ---
print("2. OpenFlamingo 모델 로딩 시작...")

model, image_processor, tokenizer = create_model_and_transforms(
    clip_vision_encoder_path="ViT-L-14",
    clip_vision_encoder_pretrained="openai",
    lang_encoder_path="anas-awadalla/mpt-1b-redpajama-200b-dolly",
    tokenizer_path="anas-awadalla/mpt-1b-redpajama-200b-dolly",
    cross_attn_every_n_layers=1
)

# 체크포인트 다운로드 및 모델 가중치 로드
checkpoint_path = hf_hub_download("openflamingo/OpenFlamingo-3B-vitl-mpt1b-langinstruct", "checkpoint.pt")
model.load_state_dict(torch.load(checkpoint_path), strict=False)

# 모델을 설정된 장치로 이동하고 평가 모드로 설정
model.to(device)
model.eval()
print("모델 로딩 및 설정 완료.")

# --- 3. 데이터 로딩 및 추론 준비 ---
print("3. CSV 데이터 로딩...")
dev_df = pd.read_csv(DEV_CSV_PATH)
submission_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)
print(f"총 {len(dev_df)}개의 샘플을 추론합니다.")

# submission_df의 ID를 인덱스로 설정하여 업데이트를 용이하게 함
submission_df.set_index('ID', inplace=True)

# 프롬프트 템플릿 정의
prompt_template = "<image>Question: {question}\nA. {A}\nB. {B}\nC. {C}\nD. {D}\nShort Answer: {choice}<|endofchunk|>"
choices = ["A", "B", "C", "D"]

# --- 4. 메인 추론 루프 ---
print("4. 추론 시작...")
print("="*50)

# dev_df를 기준으로 루프를 돌면서 추론 수행
for index, row in tqdm(dev_df.iterrows(), total=len(dev_df), desc="추론 진행률"):
    sample_id = row['ID']
    try:
        # CSV에서 데이터 추출
        img_filename = row['img_path'].split('/')[-1]
        question = row['Question']
        options = {
            "A": row['A'],
            "B": row['B'],
            "C": row['C'],
            "D": row['D']
        }

        # 전체 이미지 경로 생성
        full_img_path = os.path.join(IMAGE_DIR, img_filename)

        # 이미지 로딩 및 전처리 (5D 텐서 생성)
        query_image = Image.open(full_img_path).convert("RGB")
        processed_image = image_processor(query_image)
        vision_x = processed_image.unsqueeze(0).unsqueeze(1).unsqueeze(0)
        vision_x = vision_x.to(device)
        
        # 선택지별 Loss 계산
        losses = []
        for choice in choices:
            full_prompt = prompt_template.format(
                question=question,
                A=options["A"],
                B=options["B"],
                C=options["C"],
                D=options["D"],
                choice=choice
            )
            lang_x = tokenizer([full_prompt], return_tensors="pt")
            
            with torch.no_grad():
                outputs = model(
                    vision_x=vision_x,
                    lang_x=lang_x['input_ids'].to(device),
                    attention_mask=lang_x['attention_mask'].to(device),
                    labels=lang_x['input_ids'].clone().to(device)
                )
                losses.append(outputs.loss.item())

        # 가장 Loss가 낮은 선택지를 최종 답변으로 선택
        final_answer = choices[torch.tensor(losses).argmin()]
        
        # submission_df에 추론 결과 업데이트
        submission_df.loc[sample_id, 'answer'] = final_answer

    except Exception as e:
        print(f"\n오류 발생! 샘플 ID: {sample_id}, 오류 메시지: {e}")
        # 오류 발생 시 해당 샘플의 답변을 'Error'로 기록
        submission_df.loc[sample_id, 'answer'] = 'Error'
        continue

# --- 5. 결과 저장 ---
print("\n" + "="*50)
print("5. 모든 추론 완료. 결과를 CSV 파일로 저장합니다.")

# 인덱스를 다시 컬럼으로 변환
submission_df.reset_index(inplace=True)

# CSV 파일로 저장
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f"결과가 성공적으로 '{OUTPUT_PATH}' 파일에 저장되었습니다.")
print("="*50)

사용 장치: cuda
2. OpenFlamingo 모델 로딩 시작...
You are using config.init_device='cpu', but you can also use config.init_device="meta" with Composer + FSDP for fast initialization.
Flamingo model initialized with 1046992944 trainable parameters
모델 로딩 및 설정 완료.
3. CSV 데이터 로딩...
총 60개의 샘플을 추론합니다.
4. 추론 시작...


추론 진행률: 100%|██████████| 60/60 [00:14<00:00,  4.25it/s]


5. 모든 추론 완료. 결과를 CSV 파일로 저장합니다.
결과가 성공적으로 '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/submission/open_flamingo_2.csv' 파일에 저장되었습니다.
